# Comparing GitHub API Approaches for Release Automation

A reference notebook comparing three approaches — REST API, GraphQL API, and the `gh` CLI — for automating GitHub release workflows. Each section shows the same core operation (creating a release and uploading an asset) using a different interface, so a practitioner can evaluate trade-offs and choose the right approach for their context.

## Purpose

Release automation is a common DevOps need — tagging a commit, creating a release note, and attaching build artifacts. GitHub provides three distinct interfaces for this: the REST API, the GraphQL API, and the `gh` CLI. Each has different trade-offs in terms of flexibility, verbosity, and integration style. This notebook walks through the same release operation using all three, so the reader can compare them side by side.

## Steps

The following cells demonstrate creating a release and uploading an asset with each approach. Error handling is included to show how each interface surfaces failures.

### 1. REST API approach

The GitHub REST API uses HTTP endpoints with JSON payloads. The `requests` library in Python is a common way to interact with it. This approach is verbose but explicit — every field in the request is visible and controllable.

In [ ]:
# last_verified: 2026-08-05 · GitHub n/a
import os
import json
import requests
API_BASE = "https://api.github.com"GRAPHQL_URL = API_BASE + "/graphql"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
OWNER = "my-org"
REPO = "my-repo"
TAG = "v1.0.0"

if not GITHUB_TOKEN:
    raise EnvironmentError("GITHUB_TOKEN environment variable is not set")

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}

create_release_url = f"API_BASE + "/repos/"{OWNER}/{REPO}/releases"
release_payload = {
    "tag_name": TAG,
    "name": f"Release {TAG}",
    "body": "Automated release via REST API",
    "draft": False,
    "prerelease": False,
}

response = requests.post(create_release_url, headers=headers, json=release_payload)
if response.status_code not in (200, 201):
    print(f"Release creation failed: {response.status_code} — {response.text}")
else:
    release = response.json()
    print(f"Release created: {release['html_url']}")

upload_url = release.get("upload_url", "").replace("{?name,label}", "")
asset_path = "dist/artifact.tar.gz"

if os.path.exists(asset_path):
    with open(asset_path, "rb") as f:
        upload_response = requests.post(
            upload_url,
            headers=headers,
            params={"name": os.path.basename(asset_path)},
            data=f,
        )
        if upload_response.status_code not in (200, 201):
            print(f"Asset upload failed: {upload_response.status_code}")
        else:
            print("Asset uploaded successfully")
else:
    print(f"Asset file not found: {asset_path}")

### 2. GraphQL API approach

The GitHub GraphQL API lets you express complex operations in a single request. It is more compact than the REST API but requires understanding the schema. Mutations for release creation are supported, and the response shape can be tailored to return exactly the fields needed.

In [ ]:
# last_verified: 2026-08-05 · GitHub n/a
import os
import json
import requests

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
OWNER = "my-org"
REPO = "my-repo"
TAG = "v1.0.0"

if not GITHUB_TOKEN:
    raise EnvironmentError("GITHUB_TOKEN environment variable is not set")

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Content-Type": "application/json",
}

mutation = '''
mutation createRelease($owner: String!, $repo: String!, $tag: String!) {
  createRelease(input: {
    repositoryId: $repoId
    tagName: $tag
    name: "Release " + $tag
    body: "Automated release via GraphQL API"
    draft: false
    prerelease: false
  }) {
    release {
      url
      tagName
    }
  }
}
'''

query_url = "GRAPHQL_URL"
variables = {
    "owner": OWNER,
    "repo": REPO,
    "tag": TAG,
    "repoId": f"MDEwOlJlcG9zaXRvcnkxMzAwMTk3MDA=",
}

response = requests.post(
    query_url,
    headers=headers,
    json={"query": mutation, "variables": variables},
)
data = response.json()

if "errors" in data:
    print(f"GraphQL errors: {json.dumps(data['errors'], indent=2)}")
elif data.get("data", {}).get("createRelease", {}).get("release"):
    release = data["data"]["createRelease"]["release"]
    print(f"Release created: {release['url']}")
else:
    print("Release creation returned no result")

### 3. CLI (`gh`) approach

The `gh` CLI provides a command-line interface to GitHub's features. It is the most concise approach — a single command creates a release — and integrates naturally with shell scripts and CI pipelines. The trade-off is less flexibility for custom logic compared to the API approaches.

In [ ]:
# last_verified: 2026-08-05 · GitHub n/a
import os
import subprocess
import sys

TAG = "v1.0.0"
ASSET_PATH = "dist/artifact.tar.gz"

def run_gh(args):
    result = subprocess.run(
        ["gh"] + args,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"gh command failed: {result.stderr.strip()}")
        return None
    return result.stdout.strip()

release_args = [
    "release", "create", TAG,
    "--repo", "my-org/my-repo",
    "--title", f"Release {TAG}",
    "--notes", "Automated release via gh CLI",
]

if os.path.exists(ASSET_PATH):
    release_args.append(ASSET_PATH)
else:
    print(f"Warning: asset not found at {ASSET_PATH}, creating release without it")

output = run_gh(release_args)
if output:
    print(f"Release created: {output}")
else:
    print("Release creation returned no output")

## Verify

After running any of the three approaches, verify the release was created by listing releases for the repository. The REST API and `gh` CLI both support this directly; the GraphQL approach requires a query for the repository's releases.

In [ ]:
# last_verified: 2026-08-05 · GitHub n/a
import os
import requests

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
OWNER = "my-org"
REPO = "my-repo"

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}

url = f"API_BASE + "/repos/"{OWNER}/{REPO}/releases"
response = requests.get(url, headers=headers, params={"per_page": 5})

if response.status_code == 200:
    releases = response.json()
    for r in releases:
        print(f"{r['tag_name']}: {r['html_url']}")
else:
    print(f"Failed to list releases: {response.status_code}")